# Data Quality and Reconciliation

Task 5 quality gate - runs AFTER the Gold layer is built. Validates:

1. All eight source tables were ingested into Bronze.
2. Primary keys are populated and unique where required (Silver).
3. Gold fact table contains no null surrogate keys.
4. Numeric measures satisfy the documented business rules
   (ticket price > 0, discount in [0, 1], seats >= 1).
5. Gold revenue and row counts reconcile with Silver
   (tolerance <= 0.01%). Rejected-record counts per table are reported.

The final cell raises a pipeline-stopping error when any critical check fails.

In [0]:
from pyspark.sql import functions as F

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────
CATALOG = "sk_dev"

BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

BRONZE_TABLES = [
    "region", "nation", "airline", "passenger",
    "route", "airline_route", "booking", "flight_booking"
]

# Silver primary keys (composite keys as lists).
PRIMARY_KEY_SPECS = [
    ("region", ["region_key"]),
    ("nation", ["nation_key"]),
    ("airline", ["airline_key"]),
    ("passenger", ["passenger_key"]),
    ("route", ["route_key"]),
    ("airline_route", ["airline_key", "route_key"]),
    ("booking", ["booking_key"]),
    ("flight_booking", ["booking_key", "line_number"]),
]

FACT_TABLE = f"{GOLD_SCHEMA}.fact_bookings"

# Reconciliation tolerance: 0.01% relative difference.
RECONCILIATION_TOLERANCE = 0.0001

# ─────────────────────────────────────────────
# Check framework - every check appends a result.
# The quality gate cell raises when a CRITICAL check fails.
# ─────────────────────────────────────────────
check_results = []


def record_check(check_name, passed, details, critical=True):
    """Record and print one check result."""
    check_results.append({
        "check": check_name,
        "critical": critical,
        "passed": bool(passed),
        "details": details
    })

    status = "PASS" if passed else "FAIL"
    scope = "CRITICAL" if critical else "INFO"
    print(f"[{status}] ({scope}) {check_name} - {details}")
    return bool(passed)


def table_row_count(table_name):
    return spark.read.table(table_name).count()

In [0]:
# ─────────────────────────────────────────────
# CHECK 1 - All eight source tables ingested into Bronze
# ─────────────────────────────────────────────
print("CHECK 1 - Bronze ingestion")
print("-" * 60)

bronze_counts = {}

for table in BRONZE_TABLES:
    full_name = f"{BRONZE_SCHEMA}.{table}"

    exists = spark.catalog.tableExists(full_name)
    row_count = table_row_count(full_name) if exists else 0
    bronze_counts[table] = row_count

    record_check(
        f"Bronze ingested: {table}",
        exists and row_count > 0,
        f"exists={exists}, rows={row_count}"
    )

print()

In [0]:
# ─────────────────────────────────────────────
# CHECK 2 - Primary keys populated and unique where required
# ─────────────────────────────────────────────
print("CHECK 2 - Silver primary keys populated and unique")
print("-" * 60)

for table, keys in PRIMARY_KEY_SPECS:
    full_name = f"{SILVER_SCHEMA}.{table}"
    df = spark.read.table(full_name)
    total = df.count()

    # Populated: no NULL in any key column.
    null_condition = F.lit(False)
    for key in keys:
        null_condition = null_condition | F.col(key).isNull()
    null_key_rows = df.filter(null_condition).count()

    # Unique: distinct composite key count equals row count.
    distinct_keys = df.select(keys).distinct().count()

    record_check(
        f"Primary key populated and unique: {table} ({', '.join(keys)})",
        null_key_rows == 0 and distinct_keys == total,
        f"rows={total}, null_key_rows={null_key_rows}, "
        f"distinct_keys={distinct_keys}"
    )

print()

In [0]:
# ─────────────────────────────────────────────
# CHECK 3 - Fact table contains no null surrogate keys
# ─────────────────────────────────────────────
print("CHECK 3 - Gold fact surrogate keys")
print("-" * 60)

SURROGATE_KEYS = [
    "booking_id", "line_number", "passenger_id",
    "route_id", "airline_id"
]

fact_df = spark.read.table(FACT_TABLE)
fact_total = fact_df.count()

record_check(
    "Fact table is populated",
    fact_total > 0,
    f"rows={fact_total}"
)

for key in SURROGATE_KEYS:
    null_count = fact_df.filter(F.col(key).isNull()).count()

    record_check(
        f"Fact surrogate key not null: {key}",
        null_count == 0,
        f"rows={fact_total}, null_{key}={null_count}"
    )

print()

In [0]:
# ─────────────────────────────────────────────
# CHECK 4 - Numeric measures satisfy business rules
#   ticket_price > 0
#   discount in [0, 1] (closed interval)
#   seats >= 1
# ─────────────────────────────────────────────
print("CHECK 4 - Numeric measure business rules")
print("-" * 60)

MEASURE_RULES = [
    ("ticket_price > 0",
     F.col("ticket_price").isNull()
     | (F.col("ticket_price") <= 0)),
    ("discount in [0, 1]",
     F.col("discount").isNull()
     | (F.col("discount") < 0)
     | (F.col("discount") > 1)),
    ("seats >= 1",
     F.col("seats").isNull()
     | (F.col("seats") < 1)),
]

for rule_name, violation_condition in MEASURE_RULES:
    violations = fact_df.filter(violation_condition).count()

    record_check(
        f"Measure rule: {rule_name}",
        violations == 0,
        f"violations={violations} of {fact_total} fact rows"
    )

print()

In [0]:
# ─────────────────────────────────────────────
# CHECK 5 - Gold revenue and row counts reconcile with Silver
# Independent calculation from Silver, tolerance <= 0.01%.
# Net revenue measure: ticket_price * (1 - discount) * seats,
# rounded per line (same definition as the Gold fact table).
# ─────────────────────────────────────────────
print("CHECK 5 - Gold vs Silver reconciliation")
print("-" * 60)


def relative_difference(gold_value, silver_value):
    """Relative difference, safe against zero denominators."""
    silver_value = float(silver_value or 0)
    gold_value = float(gold_value or 0)

    if silver_value == 0:
        return 0.0 if gold_value == 0 else float("inf")

    return abs(gold_value - silver_value) / abs(silver_value)


gold_stats = spark.sql(f"""
    SELECT
        COUNT(*)         AS row_count,
        SUM(net_revenue) AS net_revenue
    FROM {FACT_TABLE}
""").first()

silver_stats = spark.sql(f"""
    SELECT
        COUNT(*)                                            AS row_count,
        SUM(ROUND(
            fb.ticket_price * (1 - fb.discount) * fb.seats, 2
        ))                                                  AS net_revenue
    FROM {SILVER_SCHEMA}.flight_booking fb
""").first()

# Row count reconciliation
row_rel_diff = relative_difference(
    gold_stats["row_count"], silver_stats["row_count"]
)
record_check(
    "Reconciliation: fact row count",
    row_rel_diff <= RECONCILIATION_TOLERANCE,
    f"Gold={gold_stats['row_count']}, Silver={silver_stats['row_count']}, "
    f"rel_diff={row_rel_diff:.8f}, tolerance={RECONCILIATION_TOLERANCE}"
)

# Net revenue reconciliation
revenue_rel_diff = relative_difference(
    gold_stats["net_revenue"], silver_stats["net_revenue"]
)
record_check(
    "Reconciliation: net revenue",
    revenue_rel_diff <= RECONCILIATION_TOLERANCE,
    f"Gold={gold_stats['net_revenue']}, "
    f"Silver={silver_stats['net_revenue']}, "
    f"rel_diff={revenue_rel_diff:.8f}, tolerance={RECONCILIATION_TOLERANCE}"
)

print()

In [0]:
# ─────────────────────────────────────────────
# Rejected-record counts per table (informational, not critical)
# ─────────────────────────────────────────────
print("Rejected-record counts per table")
print("-" * 60)

QUARANTINE_TABLES = [
    ("region", "region_quarantine"),
    ("nation", "nation_quarantine"),
    ("airline", "airline_quarantine"),
    ("passenger", "passenger_quarantine"),
    ("route", "route_quarantine"),
    ("airline_route", "airline_route_quarantine"),
    ("booking", "booking_quarantine"),
    ("flight_booking", "flight_booking_quarantine"),
]

for entity, quarantine_table in QUARANTINE_TABLES:
    full_name = f"{SILVER_SCHEMA}.{quarantine_table}"

    if spark.catalog.tableExists(full_name):
        rejected = table_row_count(full_name)
    else:
        rejected = 0

    bronze_rows = bronze_counts.get(entity, 0)
    silver_table = f"{SILVER_SCHEMA}.{entity}"
    silver_rows = (
        table_row_count(silver_table)
        if spark.catalog.tableExists(silver_table)
        else 0
    )

    record_check(
        f"Rejected records: {entity}",
        True,
        f"bronze={bronze_rows}, silver={silver_rows}, "
        f"quarantined={rejected}",
        critical=False
    )

print()

In [0]:
# ─────────────────────────────────────────────
# QUALITY GATE - pipeline-stopping error when
# any CRITICAL check fails.
# ─────────────────────────────────────────────
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

for result in check_results:
    status = "PASS" if result["passed"] else "FAIL"
    scope = "CRITICAL" if result["critical"] else "INFO"
    print(f"[{status}] ({scope}) {result['check']} - {result['details']}")

total_checks = len(check_results)
passed_checks = sum(1 for r in check_results if r["passed"])
failed_critical = [
    r for r in check_results
    if r["critical"] and not r["passed"]
]

print("-" * 60)
print(f"Checks: {passed_checks}/{total_checks} passed, "
      f"{len(failed_critical)} critical failures")
print("=" * 60)

if failed_critical:
    failure_summary = "; ".join(
        f"{r['check']} ({r['details']})"
        for r in failed_critical
    )
    raise RuntimeError(
        "DATA QUALITY GATE FAILED - pipeline stopped. "
        f"{len(failed_critical)} critical check(s) failed: "
        f"{failure_summary}"
    )

print("DATA QUALITY GATE: PASS - pipeline may continue.")